***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [9. 实践部分](9_0_introduction.ipynb)
    * 上一节：[9.38 BIMA Measurement Set 校准复盘](9_38_bima_measurement_set_calibration_replay.ipynb)
    * 下一节：[9.y 综合实践问题集](9_problem_set.ipynb)

***


## 9.39 VLA 3C391 公开归档绝对校准实验

本节把 9.35.6 的可选纵向训练落成一个经过真实运行验证的案例。输入是 NRAO/CASA 3C391 连续谱教程的 4.6 GHz Measurement Set：VLA 项目 `TDEM0001`，D 构型，2010-04-24，七场 3C391 mosaic。3C286 提供 Perley-Butler 2017 绝对通量模型，J1822-0938 提供随时间变化的复增益，再把标度转移到目标场。

本节与 9.38 的责任边界不同。9.38 使用可随仓库分发的小型 BIMA 测试 MS，训练真实列、flag、$uv$ 覆盖和相对增益，但不能建立 Jy 标度；本节有独立归档身份和可核查的外部通量模型，因而可以训练完整绝对标度链。不过，原始压缩包约 3.1 GB、展开后约 5.6 GB，且 CASA 不属于基础 Python 环境，所以默认 Notebook 只审计清单、物理模型和参考指标，不自动下载或运行大型软件。


### 9.39.1 学习目标与交付边界

完成本实验后，读者应能：核对归档身份、派生历史和再分发边界；区分初始权重、校准后逆方差权重与 flag；解释延迟、带通、时间增益和绝对通量标度的求解次序；验证通量模型和校准转移；用 beam、峰值、残差与 mask 敏感性判断成像结果；提交机器可读运行记录、QA 表和限制声明。

实验包位于 `archive_labs/vla_3c391/`。仓库只分发 GPLv2 脚本、数据 manifest 和项目实测参考指标，不分发 NRAO MS，也不把“公开下载”解释为“可按 GPLv2 再许可”。这项边界本身就是数据管理训练的一部分。


In [ ]:
import importlib.util
from pathlib import Path

import yaml

lab_dir = Path('archive_labs/vla_3c391')
if not lab_dir.exists():
    lab_dir = Path('9_Practical') / lab_dir
manifest = yaml.safe_load((lab_dir / 'manifests/data_manifest.yaml').read_text())
reference = yaml.safe_load((lab_dir / 'manifests/reference_metrics.yaml').read_text())
spec = importlib.util.spec_from_file_location('vla_3c391_audit', lab_dir / 'audit_results.py')
audit = importlib.util.module_from_spec(spec)
spec.loader.exec_module(audit)

print(manifest['sample_id'])
print(manifest['source']['archive_execution_block'])
print(f"Archive: {manifest['archive']['bytes'] / 1024**3:.2f} GiB")
print(manifest['source']['redistribution_status'])


### 9.39.2 数据状态与第一处理步骤

教程 MS 有 845,379 行、26 根天线、64 个 2 MHz 通道，覆盖 4.536--4.662 GHz，积分时间为 10 s，保留 RR、RL、LR、LL。它从原观测的 4.6 和 7.5 GHz 双谱窗中只保留前者，并从 1 s 平均到 10 s；因此它是有明确派生历史的教学起点，不是未经处理的完整归档交付。七个目标场之外，还包括 3C286、J1822-0938 和 3C84。

初始 `FLAG` 全假只表示尚未在这个 MS 中记录 flag，不表示所有数据都合格。初始 `CORRECTED_DATA` 等于 `DATA`，说明校准尚未转移；`WEIGHT_SPECTRUM` 虽有列却未初始化，起始 `WEIGHT` 也不能直接解释为可靠的 Jy$^{-2}$ 逆方差。实验因此先做人工 flag 和校准，目标拆分后才运行 `statwt`。若调换这个因果顺序，成像器得到的数值权重看似合法，物理语义却不成立。


### 9.39.3 绝对通量模型是外部证据

Perley-Butler 2017 对 3C286 使用以 GHz 为频率单位的多项式

$$\log_{10} S_\nu = \sum_{k=0}^{3} a_k[\log_{10}(\nu/\mathrm{GHz})]^k,$$

其中 $S_\nu$ 以 Jy 为单位，本实验固定 $[a_0,a_1,a_2,a_3]=[1.2481,-0.4507,-0.1798,0.0357]$。这个模型把无量纲相关系数链锚定到物理通量密度；J1822-0938 的 `fluxscale` 结果再把该标度传到时间增益。`fluxscale` 给出的统计散布不是总误差，3C286 模型系统误差、带通误差、时间插值和目标方向差异仍须单列。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

checkpoints = reference['flux_model']['checkpoints']
frequency_ghz = np.array([point['frequency_ghz'] for point in checkpoints])
expected_jy = np.array([point['flux_jy'] for point in checkpoints])
model_jy = np.array([audit.perley_butler_2017_3c286(value * 1e9) for value in frequency_ghz])
np.testing.assert_allclose(model_jy, expected_jy, atol=1e-6)

frequency_grid = np.linspace(4.53, 4.67, 200)
flux_grid = [audit.perley_butler_2017_3c286(value * 1e9) for value in frequency_grid]
plt.figure(figsize=(7, 4))
plt.plot(frequency_grid, flux_grid)
plt.scatter(frequency_ghz, expected_jy, color='tab:red', zorder=3)
plt.xlabel('Frequency [GHz]')
plt.ylabel('3C286 flux density [Jy]')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### 9.39.4 校准链与规范选择

脚本先 flag 扫描 1、天线 ea13/ea15 和每个扫描开头 10 s，并使用教程固定的天线位置偏移。随后对 3C286 运行 `setjy`，用窄通道区间求初始相位，在有效通道上求 K 型延迟和复带通，再重求 3C286 与 J1822-0938 的幅相增益。参考天线 ea21 固定了不可观测的整体相位规范。

对基线 $pq$，方向无关标量近似为

$$V_{pq}^{\mathrm{obs}}(\nu,t)=g_p(t)g_q^*(t)b_p(\nu)b_q^*(\nu)e^{-2\pi i\nu(\tau_p-\tau_q)}V_{pq}^{\mathrm{model}}+n_{pq}.$$

分步求解不是宣称这些项严格独立，而是用不同时间/频率平滑尺度限制自由度。强残余延迟会伪装成带通相位斜率，错误源模型会进入增益幅度，归一化选择还会在表之间搬移尺度。因此每张表都应检查有效解数、flag、幅相连续性和参考天线，而不能只看最终图像。标度转移后只拆分七个目标场的 RR/LL；在忽略 Stokes $V$ 且圆馈源泄漏已足够小时，Stokes $I\simeq(RR+LL)/2$。本实验不使用 RL/LR 做偏振科学。


In [ ]:
calibration = reference['calibration']
print('3C286 setjy [Jy]:', calibration['setjy_3c286_flux_jy']['value'])
print('J1822-0938 [Jy]:', calibration['gain_calibrator_flux_jy']['value'])
print('Fluxscale solutions:', calibration['valid_fluxscale_solutions'])
print('Science flag fraction:', calibration['science_flag_fraction']['value'])
print('Fully flagged antennas:', ', '.join(calibration['fully_flagged_antennas']))
print('Delay range [ns]:', calibration['delay_solution_ns']['minimum']['value'],
      calibration['delay_solution_ns']['maximum']['value'])


### 9.39.5 参考数值与回归诊断边界

CASA 6.7.0.31 验证运行得到：3C286 在 4.536 GHz 为 7.66855 Jy；J1822-0938 在 4.599 GHz 为 $2.29601\pm0.00692$ Jy，共 46 个有效标度解；有效延迟约为 $-3.85$ 到 $+4.56$ ns。目标拆分后的总 flag 比例约 34.55%，ea05、ea13、ea15 完全被 flag，七个目标场分别约为 33.5%--35.2%。

这些值可以发现数据版本、选择表达式、参考天线或任务行为的意外变化，但“落在容差内”并不自动证明校准可信。学生还应绘制随天线、时间和通道变化的解，检查断点、相位绕转、孤立天线和校准源残差；任何额外 flag 都必须说明证据和对约束图的影响。


### 9.39.6 成像基线中的受控失败模式

参考成像使用 480$\times$480 像元、2.5 arcsec 像元、mosaic gridder、Briggs robust 0.5 和多尺度 `[0,5,15,45]`。验证运行的恢复波束约为 $17.0\times14.9$ arcsec，位置角约 $21.0^\circ$；dirty 峰值约 0.1229 Jy beam$^{-1}$，恢复图峰值约 0.1301 Jy beam$^{-1}$，全图残差 RMS 约 0.570 mJy beam$^{-1}$，残差极值约为 $-2.27$ 到 $+2.24$ mJy beam$^{-1}$。固定 130 像元大圆 mask 在 minor cycle 中产生 11 次大尺度负分量或发散警告，模型总和约 3.788 Jy，其中负分量占模型绝对通量约 25.6%。

受控实验保持 visibility、网格、权重、阈值和恢复波束不变。100 像元紧圆把负分量比例降到 12.9%，但残差 RMS 升至 0.840 mJy beam$^{-1}$，mask 边界出现强弧状残差；保守 `auto-multithresh` 把负分量降到 0.7%，却把模型总和推到 8.070 Jy，产生 17 次发散警告，残差 RMS 仍为 0.645 mJy beam$^{-1}$，且 45 像元尺度曾因无法放入局部 mask 而被忽略。保持大圆、只删除 45 像元尺度可消除发散警告，但残差 RMS 为 0.737 mJy beam$^{-1}$，一个波束尺度的相关系数升至约 0.85/0.90，说明扩展结构主要被留在残差中。

因此四组方案没有一组支持更强的科学产品。更少的负 CLEAN 分量、较低 RMS 或没有发散警告都只是单项证据，不能单独决定优劣；mask 还会改变有效模型空间，使模型通量不能被当作彼此独立的同一估计量。固定大圆只保留为可复现回归基线，发表级结论仍为 `false`。后续若要测量目标通量，必须进一步检查 visibility 与 PSF 对大尺度的约束、独立审查源支撑区，并处理主波束、相关噪声和 mosaic 空间响应。


In [ ]:
imaging = reference['imaging']
beam = imaging['restoring_beam']
print('Beam [arcsec]:', beam['major_arcsec']['value'], 'x', beam['minor_arcsec']['value'])
print('Restored peak [Jy/beam]:', imaging['restored_peak_jy_per_beam']['value'])
print('Residual RMS [mJy/beam]:', 1e3 * imaging['residual_rms_jy_per_beam']['value'])

sensitivity = reference['imaging_sensitivity']
for name in ('fixed_broad', 'fixed_tight', 'auto_conservative',
             'fixed_broad_restricted_scales'):
    result = sensitivity[name]
    print(name, 'model [Jy]:', result['model_sum_jy'],
          'residual [mJy/beam]:', result['residual_rms_mjy_per_beam'])
print('Sensitivity conclusion:', sensitivity['conclusion']['status'])
print('Publication claim:', sensitivity['conclusion']['publication_claim_supported'])


### 9.39.7 课程纵向训练（100 分）

1. **归档与许可（10 分）**：核对项目、执行块、观测日期、阵列构型、教程派生历史、URL、字节数和 SHA-256；写明为何本仓库不再分发 MS。
2. **初始 MS 审计（10 分）**：报告主表行数、场、天线、谱窗、通道、相关积、积分时间和关键列；证明初始 `FLAG`、`CORRECTED_DATA` 与权重分别允许或不允许什么结论。
3. **Flag 证据（10 分）**：按扫描、天线、场、时间和通道比较 flag 前后统计，解释 ea05/ea13/ea15 的处理怎样改变求解约束。
4. **绝对标度（15 分）**：复现 3C286 多项式，核对 `setjy` 和 J1822-0938 `fluxscale`，把形式统计误差与通量模型、转移和方向系统误差分开。
5. **校准表 QA（20 分）**：检查 G0、K0、B0、G1 与 fluxscale 表的有效解、幅相连续性、频率结构、参考天线和校准源残差；至少给出一个失败判据。
6. **转移与权重（10 分）**：说明目标 gainfield/interpolation 选择，拆分 RR/LL 后运行 `statwt`；验证权重和 flag 与数据形状一致。
7. **成像与失败模式（15 分）**：运行 `run_imaging_sensitivity.py` 或等价受控实验，比较 mask 面积、模型正负通量、残差 RMS/极值/空间相关、有效尺度和停止日志；说明为何本节四个实测方案都不应升级为发表产品，并提出下一项能区分成因的检验。
8. **复现报告（10 分）**：提交软件版本、CASA 数据路径配置、命令、参数、运行日志、JSON 审计报告、产品校验和与限制声明。

评分依据是证据链，不是最终图像是否好看。校准或成像未通过时，正确停止、定位失败并降低结论等级可以获得相应判断分；隐去错误或用参考数值替代自己的输出不能得分。


### 9.39.8 运行与审计

`download_data.py` 支持断点续传、长度与 SHA-256 校验，并在解包前拒绝越界路径、链接和设备文件。`run_casa_pipeline.py` 把输入复制到空工作目录，分为 calibration 和 imaging 两个停止点，保存 CASA 版本、参数、flag、fluxscale、延迟、statwt、成像统计和恢复波束。`run_imaging_sensitivity.py` 复用校准后的目标 MS，在独立空目录中运行三种 mask 和一项尺度诊断，输出 `imaging_sensitivity.json`；它不会重复约 6 GB 的校准链。`audit_results.py` 用带容差的参考指标生成通过/失败报告；`--require-imaging` 可要求完整链。详细命令见实验包 [README](archive_labs/vla_3c391/README.md)。

参考指标只用于验证同一输入和同一基线流程是否一致。更换 CASA 版本、flag、通量模型、参考天线、mask 或成像参数后，应保留新的运行身份并解释差异，不应为了“通过测试”把科学上合理的变化硬调回旧值。反过来，任何新流程也必须重新建立可核查的数值基线。


### 9.39.9 本节结论

3C391 案例补齐了从公开归档身份、外部绝对通量模型、延迟/带通/时间增益到 mosaic 成像和最终 QA 的纵向训练。它证明教材已经准备好一套经过真实数据验证的课程实验入口，但没有把大型第三方数据、CASA 或单一参考图像变成基础学习的前置条件。

更重要的是，实验没有把“流水线运行结束”等同于“科学结论完成”。绝对标度必须追溯到外部模型，权重必须在校准后的数据语义下重估，成像必须报告 mask 与残差失败模式。四组受控实验没有产生明确更好的图像，这不是需要隐藏的失败，而是可评分的停止决定：不同 mask 和尺度在负模型、通量、发散与相关残差之间交换风险，现有证据不足以支持发表级测量。参考指标必须与科学验收区分，完成这种结论降级，才算真正完成一次可审查的射电干涉数据处理复盘。

***
